# CS781 Phase 2

In [1]:
%%writefile lner.py
#!/usr/bin/env python3
"""
lner.py  --  CS781 Phase 2 (Legal NER) pipeline on top of YOUR Phase-1 RoPE+MQA+SwiGLU decoder.

One run = one adaptation method.  Examples:
  python lner.py --method full  --attn bidir  --out runs/full_bidir
  python lner.py --method lorma --out runs/lorma
  python lner.py --method kd --teacher runs/full_bidir/best.pt --out runs/kd

Sections
  A. Phase-1 tokenizer + model code (copied from your notebooks; the ONLY edits are marked  # [P2-EDIT])
  B. Offset-exact tokenisation, BIO labelling, windowing
  C. Adaptation methods (full / partial / bitfit / lora / lorma / lorma_pi / adapter / ia3 / kd)
  D. Training, decoding (greedy / BIO-constrained Viterbi), strict evaluation, submission writer
"""
import argparse, hashlib, json, math, os, pickle, random, re, sys, time, zipfile
from bisect import bisect_left
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IGN = -100
PAD_ID = 256          # "<pad>" in your tokenizer (256 + index in SPECIAL_TOKENS)
EOS_ID = 259          # "</s>"  (word separator in your Phase-1 tokenizer)


class ByteLevelBPETokenizer:
    SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            tid = 256 + i
            self.special_to_id[tok] = tid
            self.id_to_bytes[tid] = tok.encode("utf-8")

    @property
    def vocab_size(self):
        return len(self.id_to_bytes)

    @staticmethod
    def _pair_counts(sequences):
        counts = {}
        for seq, freq in sequences:
            for pair in zip(seq, seq[1:]):
                counts[pair] = counts.get(pair, 0) + freq
        return counts

    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                out.append(new_id); i += 2
            else:
                out.append(seq[i]); i += 1
        return out

    def train(self, text, vocab_size, max_words=100_000, verbose=False):   # only used by the smoke test
        num_merges = vocab_size - self.vocab_size
        word_freq = {}
        for word in text.split():
            word_freq[word] = word_freq.get(word, 0) + 1
        sequences = [[list(w.encode("utf-8")), f] for w, f in word_freq.items()]
        next_id = self.vocab_size
        for _ in range(num_merges):
            counts = self._pair_counts(sequences)
            if not counts: break
            best = max(counts, key=counts.get)
            self.merges[best] = next_id
            self.id_to_bytes[next_id] = self.id_to_bytes[best[0]] + self.id_to_bytes[best[1]]
            for e in sequences: e[0] = self._apply_merge(e[0], best, next_id)
            next_id += 1
        return self

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": {f"{a},{b}": n for (a, b), n in self.merges.items()}}, f)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1 = x[..., ::2]; x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, q, k):
        T = q.size(-2)
        cos = self.cos[:, :, :T, :]; sin = self.sin[:, :, :T, :]
        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class MultiQueryCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % kv_heads == 0
        self.n_heads, self.kv_heads = n_heads, kv_heads
        self.d_head = d_model // n_heads
        self.kv_dim = kv_heads * self.d_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.d_head, max_seq_len=max_seq_len, base=rope_base)
        self.dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.causal = True                                   # [P2-EDIT] switch for bidirectional fine-tuning

    def forward(self, x, pad_mask=None):                     # [P2-EDIT] optional key padding mask
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        q, k = self.rope(q, k)
        if self.kv_heads != self.n_heads:
            r = self.n_heads // self.kv_heads
            k = k.repeat_interleave(r, dim=1); v = v.repeat_interleave(r, dim=1)
        dp = self.dropout if self.training else 0.0
        if self.causal:                                      # right padding is harmless under a causal mask
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=dp, is_causal=True)
        else:
            m = None if pad_mask is None else pad_mask[:, None, None, :]   # True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, dropout_p=dp, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.silu(self.gate(x)) * self.up(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.attn = MultiQueryCausalAttention(d_model, n_heads, kv_heads, max_seq_len, dropout, rope_base)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, pad_mask=None):                     # [P2-EDIT] pass the mask through
        x = x + self.attn(self.norm_attn(x), pad_mask)
        x = x + self.ffn(self.norm_ffn(x))
        return x


class RoPEMQADecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, kv_heads, d_ff, max_seq_len,
                 dropout=0.1, rope_base=10000.0, tie_weights=True):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base)
                                     for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_emb.weight


DEFAULT_LM_CFG = dict(vocab_size=12000, d_model=768, n_layers=12, n_heads=12, kv_heads=1,
                      d_ff=3072, max_seq_len=512, dropout=0.1, rope_base=10000.0)


def build_lm(cfg):
    c = {**DEFAULT_LM_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_LM_CFG}}
    return RoPEMQADecoderLM(**c, tie_weights=True), c


def load_phase1(path, n_layers=None, layer_map=None):
    """Load your Phase-1 weights.  Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config})
    or a raw state_dict (model_stepXXXXX.pt).  Optionally keeps only `layer_map` blocks (for the KD student)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], ck.get("config", {})
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]
        rc = ck.get("run_config", {})
        cfg = dict(vocab_size=rc.get("VOCAB_SIZE"), d_model=rc.get("D_MODEL"), n_layers=rc.get("N_LAYERS"),
                   n_heads=rc.get("N_HEADS"), d_ff=rc.get("D_FF"), max_seq_len=rc.get("BLOCK_SIZE"),
                   kv_heads=rc.get("KV_HEADS"), rope_base=rc.get("ROPE_BASE"))
        cfg = {k: v for k, v in cfg.items() if v is not None}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    cfg["n_layers"] = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["blocks.0.ffn.gate.weight"].shape[0]
    cfg["kv_heads"] = sd["blocks.0.attn.k_proj.weight"].shape[0] // (cfg["d_model"] // cfg.get("n_heads", 12))
    if layer_map is not None:                                # layer-dropped student: blocks.{old} -> blocks.{new}
        new = {k: v for k, v in sd.items() if not k.startswith("blocks.")}
        for ni, oi in enumerate(layer_map):
            for k, v in sd.items():
                if k.startswith(f"blocks.{oi}."):
                    new[f"blocks.{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_layers"] = new, len(layer_map)
    lm, cfg = build_lm(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k != "lm_head.weight"] + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg


WORD_RE = re.compile(r"\S+")             # same segmentation as str.split() used in Phase 1
PIECE_RE = re.compile(r"\w+|[^\w\s]")    # punctuation is split off so entity edges can sit next to it


class Piecer:
    """BPE-encodes one piece at a time and returns char offsets for every BPE token (cached)."""
    def __init__(self, tok):
        self.tok, self.cache = tok, {}

    def piece(self, s):
        r = self.cache.get(s)
        if r is None:
            tok = self.tok
            seq = list(s.encode("utf-8"))
            while len(seq) > 1:
                mergeable = [p for p in zip(seq, seq[1:]) if p in tok.merges]
                if not mergeable: break
                pair = min(mergeable, key=lambda p: tok.merges[p])
                seq = tok._apply_merge(seq, pair, tok.merges[pair])
            b2c = []
            for ci, ch in enumerate(s):
                b2c.extend([ci] * len(ch.encode("utf-8")))
            offs, b = [], 0
            for t in seq:
                n = len(tok.id_to_bytes[t])
                offs.append((b2c[b], b2c[b + n - 1] + 1)); b += n
            r = (tuple(seq), tuple(offs))
            self.cache[s] = r
        return r

    def encode_doc(self, text):
        ids, cs, ce = [], [], []
        for wm in WORD_RE.finditer(text):
            w, ws = wm.group(), wm.start()
            for pm in PIECE_RE.finditer(w):
                tids, offs = self.piece(pm.group())
                base = ws + pm.start()
                for t, (a, b) in zip(tids, offs):
                    ids.append(t); cs.append(base + a); ce.append(base + b)
            ids.append(EOS_ID); cs.append(-1); ce.append(-1)      # word separator, exactly as in pre-training
        return np.array(ids, np.int32), np.array(cs, np.int32), np.array(ce, np.int32)


class LabelSpace:
    def __init__(self, labels):
        self.labels = list(labels)
        self.k = {l: i for i, l in enumerate(self.labels)}
        self.n = 1 + 2 * len(self.labels)                       # O, B-x, I-x ...
        self.names = ["O"] + [f"{p}-{l}" for l in self.labels for p in ("B", "I")]

    def b(self, label): return 1 + 2 * self.k[label]
    def i(self, label): return 2 + 2 * self.k[label]


def spans_from_ids(lab):
    """Lenient BIO -> [(first_tok, last_tok, label_idx)] over a sequence of REAL tokens."""
    out, cur = [], None
    for t, y in enumerate(lab):
        if y == 0:
            if cur: out.append(tuple(cur)); cur = None
        else:
            k, is_b = (y - 1) // 2, (y - 1) % 2 == 0
            if is_b or cur is None or cur[2] != k:
                if cur: out.append(tuple(cur))
                cur = [t, t, k]
            else:
                cur[1] = t
    if cur: out.append(tuple(cur))
    return out


def parse_gold(raw, text):
    """Accepts [[s,e,label(,txt)], ...] or [{'start','end','label','text'}, ...]. Returns [(s,e,label,txt|None)]."""
    out = []
    for ent in raw or []:
        if isinstance(ent, dict):
            g = lambda *ks: next((ent[k] for k in ks if k in ent), None)
            s, e, l, t = g("start", "start_char", "begin"), g("end", "end_char"), g("label", "type", "entity"), g("text", "entity_text", "span")
        else:
            s, e, l = ent[0], ent[1], ent[2]
            t = ent[3] if len(ent) > 3 else None
        out.append((int(s), int(e), l, t))
    return out


def detect_end_shift(texts, golds, n=3000):
    """0 if gold end is exclusive (text[s:e]==entity_text), 1 if inclusive. Fails loudly if neither matches."""
    ex = inc = tot = 0
    for did, ents in golds.items():
        for s, e, l, t in ents:
            if t is None: continue
            tot += 1
            ex += texts[did][s:e] == t
            inc += texts[did][s:e + 1] == t
            if tot >= n: break
        if tot >= n: break
    if tot == 0:
        print("  [offsets] gold has no entity_text field -> assuming end is EXCLUSIVE (python slice)."); return 0
    print(f"  [offsets] text[s:e]==entity_text for {ex}/{tot};  text[s:e+1]==entity_text for {inc}/{tot}")
    if ex / tot > 0.9: return 0
    if inc / tot > 0.9: return 1
    raise RuntimeError("Gold offsets do not reproduce entity_text under either convention. Inspect a few examples "
                       "(maybe the text was normalised?) before training.")


def assign_bio(cs, ce, ents, ls):
    """Token-level labels (len == #tokens incl. </s>, IGN on </s>). Returns (labels, n_overlap, n_unaligned)."""
    real = np.where(cs >= 0)[0]
    rs, re_ = cs[real], ce[real]
    y = np.full(len(cs), IGN, np.int16)
    lab_real = np.zeros(len(real), np.int16)
    n_over = n_unal = 0
    for s, e, l, _ in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = int(np.searchsorted(rs, s, "left"))
        hi = int(np.searchsorted(re_, e, "right"))
        if hi <= lo: n_unal += 1; continue
        if (lo > 0 and re_[lo - 1] > s) or (hi < len(rs) and rs[hi] < e) or rs[lo] != s or re_[hi - 1] != e:
            n_unal += 1          # entity edge falls inside a token -> cannot be reproduced exactly
        if (lab_real[lo:hi] != 0).any():
            n_over += 1; continue
        lab_real[lo] = ls.b(l); lab_real[lo + 1:hi] = ls.i(l)
    y[real] = lab_real
    return y, n_over, n_unal


def make_windows(n, W, stride):
    if n <= W: return [(0, n)]
    out, a = [], 0
    while True:
        b = min(a + W, n)
        out.append((a, b))
        if b == n: break
        a += stride
    return out


def to_char_spans(lab_real, cs_real, ce_real, ls):
    return [(int(cs_real[i]), int(ce_real[j]), ls.labels[k]) for i, j, k in spans_from_ids(lab_real)]


def strict_scores(truth, pred, labels):
    st = {l: [0, 0, 0] for l in labels}
    for did in truth:
        g = {tuple(x[:3]) for x in truth[did]}; p = {tuple(x[:3]) for x in pred.get(did, [])}
        for l in labels:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            st[l][0] += len(gs & ps); st[l][1] += len(ps - gs); st[l][2] += len(gs - ps)
    per, P, R, F1 = {}, [], [], []
    for l in labels:
        tp, fp, fn = st[l]
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = dict(P=pr, R=rc, F1=f, tp=tp, fp=fp, fn=fn); P.append(pr); R.append(rc); F1.append(f)
    return dict(macro_F1=sum(F1) / len(F1), macro_P=sum(P) / len(P), macro_R=sum(R) / len(R), per_label=per)


def error_breakdown(truth, pred, labels):
    """What kind of mistakes? exact / boundary (same label, overlapping) / label confusion (same span) / spurious / missed."""
    c = dict(exact=0, boundary=0, label_confusion=0, spurious=0, missed=0)
    conf = {}
    for did in truth:
        g = [tuple(x[:3]) for x in truth[did]]; p = [tuple(x[:3]) for x in pred.get(did, [])]
        gset, pset = set(g), set(p)
        for (s, e, l) in p:
            if (s, e, l) in gset: c["exact"] += 1; continue
            same_span = [x for x in g if x[0] == s and x[1] == e]
            ovl_same = [x for x in g if x[2] == l and x[0] < e and s < x[1]]
            ovl_any = [x for x in g if x[0] < e and s < x[1]]
            if same_span: c["label_confusion"] += 1; conf[(same_span[0][2], l)] = conf.get((same_span[0][2], l), 0) + 1
            elif ovl_same: c["boundary"] += 1
            elif ovl_any: c["label_confusion"] += 1
            else: c["spurious"] += 1
        for (s, e, l) in g:
            if (s, e, l) not in pset and not any(x[0] < e and s < x[1] for x in p): c["missed"] += 1
    top = sorted(conf.items(), key=lambda kv: -kv[1])[:8]
    c["top_label_confusions(gold->pred)"] = [[a, b, n] for (a, b), n in top]
    return c


class NERModel(nn.Module):
    def __init__(self, lm, num_labels, causal=True, head_dropout=0.1, head="linear"):
        super().__init__()
        self.token_emb, self.blocks, self.final_norm = lm.token_emb, lm.blocks, lm.final_norm
        D = self.token_emb.embedding_dim
        self.drop = nn.Dropout(head_dropout)
        self.head = nn.Linear(D, num_labels) if head == "linear" else nn.Sequential(
            nn.Linear(D, D), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(D, num_labels))
        self.set_causal(causal)

    def set_causal(self, causal):
        self.causal = causal
        for b in self.blocks: b.attn.causal = causal

    def forward(self, idx, pad_mask=None):
        x = self.token_emb(idx)
        for b in self.blocks: x = b(x, pad_mask)
        return self.head(self.drop(self.final_norm(x)))


class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) B A x   (Hu et al. 2021)"""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMAPlusLinear(nn.Module):
    """LoRMA_+ (Bihany et al. 2025, arXiv 2506.07621): W' = ((alpha/r) B A + I) W0  -- MULTIPLICATIVE on the output side.
    Right-to-left evaluation:  h = W0 x ;  y = h + (alpha/r) B (A h).   B=0, A random -> identity at init."""
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))

    def forward(self, x):
        h = self.base(x)
        return h + self.s * F.linear(F.linear(h, self.A), self.B)


class LoRMAPiLinear(nn.Module):
    """LoRMA_pi: W' = I_pi(B A) W0 where I_pi cyclically rotates row i of BA by i.  Init: B[:,0]=1, A[0,0]=1, rest of A=0
    -> rot(BA)=I.  (Scale of the 'random' part of B is NOT specified in what I read; std=0.02 is an assumption.)"""
    def __init__(self, base, r):
        super().__init__()
        d = base.out_features
        self.base = base
        B = torch.randn(d, r) * 0.02; B[:, 0] = 1.0
        A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        ar = torch.arange(d)
        self.register_buffer("idx", (ar[None, :] - ar[:, None]) % d, persistent=False)   # M[i,j] = P[i,(j-i) mod d]

    def forward(self, x):
        h = self.base(x)
        M = torch.gather(self.B @ self.A, 1, self.idx)
        return F.linear(h, M.to(h.dtype))


class AdapterWrap(nn.Module):
    """Houlsby-style bottleneck adapter applied to the sub-layer output (before the residual add)."""
    def __init__(self, inner, d, bottleneck):
        super().__init__()
        self.inner, self.down, self.up = inner, nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        y = self.inner(*a, **k)
        return y + self.up(F.gelu(self.down(y)))


class IA3Out(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x): return self.base(x) * self.l


class IA3In(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x): return self.base(x * self.l)


ATTN_T = ("q_proj", "k_proj", "v_proj", "out_proj")
FFN_T = ("gate", "up", "down")


def apply_method(model, args):
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for p in model.head.parameters(): p.requires_grad = True                      # the new head is always trained
    D = model.token_emb.embedding_dim
    tg = [t.strip() for t in args.targets.split(",")]

    def wrap_linears(factory, names):
        for b in model.blocks:
            for t in names:
                parent = b.attn if t in ATTN_T else b.ffn
                setattr(parent, t, factory(getattr(parent, t)))

    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "partial":                                                          # layer-wise: top-k blocks + final norm
        for b in model.blocks[-args.top_layers:]:
            for p in b.parameters(): p.requires_grad = True
        for p in model.final_norm.parameters(): p.requires_grad = True
    elif m == "bitfit":                                                           # model has no biases -> tune RMSNorm gains
        for mod in model.modules():
            if isinstance(mod, RMSNorm):
                for p in mod.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: LoRALinear(l, args.rank, args.alpha, args.lora_dropout), tg)
    elif m == "lorma":
        wrap_linears(lambda l: LoRMAPlusLinear(l, args.rank, args.alpha), tg)
    elif m == "lorma_pi":
        wrap_linears(lambda l: LoRMAPiLinear(l, args.rank), [t for t in tg if t in ATTN_T])   # d_out<=768 only (cheap)
    elif m == "adapter":
        for b in model.blocks:
            b.attn = AdapterWrap(b.attn, D, args.adapter_dim)
            b.ffn = AdapterWrap(b.ffn, D, args.adapter_dim)
    elif m == "ia3":
        for b in model.blocks:
            b.attn.k_proj, b.attn.v_proj = IA3Out(b.attn.k_proj), IA3Out(b.attn.v_proj)
            b.ffn.down = IA3In(b.ffn.down)
    else:
        raise ValueError(m)
    return model


def mark_new_params_trainable(model, before_ids):
    """Parameters created by a wrapper (LoRA/LoRMA/adapter/IA3) are trainable; wrapped base weights keep their identity."""
    for p in model.parameters():
        if id(p) not in before_ids: p.requires_grad = True


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


def load_split_files(data_dir):
    def rd(name):
        with open(os.path.join(data_dir, name), encoding="utf-8") as f: return json.load(f)
    with open(os.path.join(data_dir, "labels.txt"), encoding="utf-8") as f:
        labels = [x.strip() for x in f if x.strip()]
    texts, golds = {}, {}
    for fold in ("fold1", "fold2"):
        t, g = rd(f"{fold}-text.json"), rd(f"{fold}-gold.json")
        if isinstance(g, list):                                   # tolerate [{"id":..,"entities":[..]}, ...]
            g = {str(x.get("id")): x.get("entities", x.get("annotations", x.get("labels", []))) for x in g}
        g = {str(k): v for k, v in g.items()}
        for did, txt in t.items():
            did = str(did)
            texts[did] = txt
            golds[did] = parse_gold(g.get(did), txt)
    test = {str(k): v for k, v in rd("fold3-text.json").items()}
    return labels, texts, golds, test


def build_docs(texts, golds, ls, piecer, shift, tag=""):
    docs = []
    n_ent = n_over = n_unal = 0
    for did, txt in texts.items():
        ids, cs, ce = piecer.encode_doc(txt)
        ents = [(s, e + shift, l, t) for s, e, l, t in golds.get(did, [])]
        bad = [x for x in ents if x[2] not in ls.k]
        assert not bad, f"gold label not in labels.txt: {bad[:3]}"
        y, no, nu = assign_bio(cs, ce, ents, ls)
        n_ent += len(ents); n_over += no; n_unal += nu
        docs.append(dict(id=did, ids=ids, cs=cs, ce=ce, y=y, ents=[(s, e, l) for s, e, l, _ in ents]))
    print(f"  [{tag}] docs={len(docs)} tokens={sum(len(d['ids']) for d in docs):,} entities={n_ent:,} "
          f"overlap-dropped={n_over} edge-inside-token={n_unal} ({100 * n_unal / max(1, n_ent):.2f}%)")
    return docs


def oracle_f1(docs, ls):
    """Decode the GOLD token labels back to char spans: this is the ceiling of the whole tokenisation/alignment pipeline."""
    truth, pred = {}, {}
    for d in docs:
        real = d["cs"] >= 0
        pred[d["id"]] = to_char_spans(d["y"][real], d["cs"][real], d["ce"][real], ls)
        truth[d["id"]] = list(set(d["ents"]))
    return strict_scores(truth, pred, ls.labels)["macro_F1"]


def collate(items, device):
    L = max(len(x[0]) for x in items)
    ids = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(items), L), IGN, dtype=torch.long)
    msk = torch.zeros((len(items), L), dtype=torch.bool)
    for i, (a, b) in enumerate(items):
        ids[i, :len(a)] = torch.from_numpy(a.astype(np.int64)); msk[i, :len(a)] = True
        if b is not None: lab[i, :len(b)] = torch.from_numpy(b.astype(np.int64))
    return ids.to(device), lab.to(device), (None if msk.all() else msk.to(device))


def train_windows(docs, W, stride, neg_keep, rng):
    out = []
    for di, d in enumerate(docs):
        for a, b in make_windows(len(d["ids"]), W, stride):
            has = (d["y"][a:b] > 0).any()
            if has or rng.random() < neg_keep: out.append((di, a, b))
    return out


@torch.no_grad()
def doc_logprobs(model, doc, W, stride, device, bs=16, amp=True):
    n = len(doc["ids"])
    if n == 0: return np.zeros((0, model_nl(model)), np.float32)
    acc = np.zeros((n, model_nl(model)), np.float32); wsum = np.zeros(n, np.float32)
    wins = make_windows(n, W, stride)
    causal = model.causal
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        ids, _, msk = collate([(doc["ids"][a:b], None) for a, b in chunk], device)
        with torch.autocast("cuda", enabled=amp and device.type == "cuda"):
            lg = model(ids, msk)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for j, (a, b) in enumerate(chunk):
            L = b - a; pos = np.arange(L, dtype=np.float32)
            w = pos + 1 if causal else np.minimum(pos + 1, L - pos)
            acc[a:b] += lp[j, :L] * w[:, None]; wsum[a:b] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def model_nl(model):
    h = model.head
    return h.out_features if isinstance(h, nn.Linear) else h[-1].out_features


def viterbi(lp, ls):
    T, N = lp.shape
    trans = np.zeros((N, N), np.float32)
    for j in range(1, N):
        if (j - 1) % 2 == 1:                                    # I-x only after B-x / I-x of the same type
            k = (j - 1) // 2
            trans[:, j] = -1e4; trans[1 + 2 * k, j] = 0; trans[2 + 2 * k, j] = 0
    start = np.zeros(N, np.float32); start[[j for j in range(1, N) if (j - 1) % 2 == 1]] = -1e4
    score = lp[0] + start
    bp = np.zeros((T, N), np.int16)
    for t in range(1, T):
        cand = score[:, None] + trans
        bp[t] = cand.argmax(0); score = cand.max(0) + lp[t]
    path = np.zeros(T, np.int64); path[-1] = score.argmax()
    for t in range(T - 1, 0, -1): path[t - 1] = bp[t, path[t]]
    return path


def predict_docs(model, docs, ls, W, stride, device, decodes=("viterbi",), shift=0, amp=True):
    model.eval()
    out = {d: {} for d in decodes}; n_tok = 0; t0 = time.time()
    for doc in docs:
        lp = doc_logprobs(model, doc, W, stride, device, amp=amp)
        real = doc["cs"] >= 0
        lpr, csr, cer = lp[real], doc["cs"][real], doc["ce"][real]
        n_tok += len(doc["ids"])
        for dm in decodes:
            if len(lpr) == 0: out[dm][doc["id"]] = []; continue
            lab = viterbi(lpr, ls) if dm == "viterbi" else lpr.argmax(-1)
            out[dm][doc["id"]] = to_char_spans(lab, csr, cer, ls)
    return out, n_tok / max(1e-6, time.time() - t0)


@torch.no_grad()
def collect_logp(model, docs, W, stride, device, amp=True):
    """Window-merged token log-probs (REAL tokens only, float16) per doc -> saved for ensembling / post-processing."""
    model.eval(); out = {}; n_tok = 0; t0 = time.time()
    for doc in docs:
        lp = doc_logprobs(model, doc, W, stride, device, amp=amp)
        out[doc["id"]] = lp[doc["cs"] >= 0].astype(np.float16); n_tok += len(doc["ids"])
    return out, n_tok / max(1e-6, time.time() - t0)


def decode_logp(lp_dict, docs, ls, mode):
    out = {}
    for d in docs:
        lpr = lp_dict[d["id"]].astype(np.float32); real = d["cs"] >= 0
        if len(lpr) == 0: out[d["id"]] = []; continue
        lab = viterbi(lpr, ls) if mode == "viterbi" else lpr.argmax(-1)
        out[d["id"]] = to_char_spans(lab, d["cs"][real], d["ce"][real], ls)
    return out


def dev_truth_of(docs):
    return {d["id"]: sorted(set(d["ents"])) for d in docs}


def shifted(preds, shift):
    return {k: [[s, e - shift, l] for s, e, l in v] for k, v in preds.items()}


def lr_at(step, total, warm, base):
    if step < warm: return base * (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return base * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


def save_ckpt(path, model, meta):
    torch.save({"state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "meta": meta}, path)


def load_ner_ckpt(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck["meta"]
    lm, _ = build_lm(meta["lm_cfg"])
    ls = LabelSpace(meta["labels"])
    model = NERModel(lm, ls.n, causal=meta["causal"], head=meta.get("head", "linear"))
    assert meta["method"] in ("full", "kd", "partial", "bitfit"), "teacher must be a method without structural wrappers"
    model.load_state_dict(ck["state"]); return model.to(device).eval(), meta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True); ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--init", required=True, help="Phase-1 weights (model_step45000.pt / model.pt / checkpoint.pt)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--method", required=True, choices=["full", "partial", "bitfit", "lora", "lorma", "lorma_pi", "adapter", "ia3", "kd"])
    ap.add_argument("--attn", default="bidir", choices=["causal", "bidir"])
    ap.add_argument("--head", default="linear", choices=["linear", "mlp"])
    ap.add_argument("--epochs", type=float, default=3.0); ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=None); ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.01); ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--window", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--targets", default="q_proj,k_proj,v_proj,out_proj,gate,up,down")
    ap.add_argument("--adapter_dim", type=int, default=64); ap.add_argument("--top_layers", type=int, default=4)
    ap.add_argument("--o_weight", type=float, default=0.5, help="loss weight of the 'O' class (class balancing)")
    ap.add_argument("--neg_keep", type=float, default=0.5, help="fraction of entity-free windows kept per epoch")
    ap.add_argument("--dev_frac", type=float, default=0.06); ap.add_argument("--dev_docs", type=int, default=200)
    ap.add_argument("--full_data", action="store_true", help="also train on the dev docs (use for final runs only)")
    ap.add_argument("--time_budget_min", type=float, default=90.0)
    ap.add_argument("--teacher", default=None); ap.add_argument("--student_layers", type=int, default=6)
    ap.add_argument("--kd_T", type=float, default=2.0); ap.add_argument("--kd_alpha", type=float, default=0.5)
    ap.add_argument("--max_train_docs", type=int, default=0); ap.add_argument("--no_test", action="store_true")
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--cache", default=None)
    ap.add_argument("--n_folds", type=int, default=0, help="k-fold CV over the 70 labelled docs (0 = old single dev split)")
    ap.add_argument("--fold", type=int, default=0, help="which fold is the dev fold when --n_folds > 0")
    args = ap.parse_args()

    t_global = time.time()
    os.makedirs(args.out, exist_ok=True)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    default_lr = dict(full=1e-4, kd=1e-4, partial=1e-4, bitfit=2e-3, lora=1e-3, lorma=1e-3, lorma_pi=5e-4, adapter=1e-3, ia3=3e-3)
    base_lr = args.lr if args.lr is not None else default_lr[args.method]
    print(f"== method={args.method} attn={args.attn} lr={base_lr} device={device}")

    tok = ByteLevelBPETokenizer.load(args.tokenizer)
    labels, texts, golds, test_texts = load_split_files(args.data_dir)
    assert len(labels) == 12, labels
    ls = LabelSpace(labels)
    shift = detect_end_shift(texts, golds)
    key = hashlib.md5((open(args.tokenizer, "rb").read()[:100000] + str(shift).encode() + str(len(texts)).encode())).hexdigest()[:10]
    cache = args.cache or os.path.join(args.data_dir, f"tokcache_{key}.pkl")
    piecer = Piecer(tok)
    if os.path.exists(cache):
        with open(cache, "rb") as f: all_docs, test_docs = pickle.load(f)
        print(f"  loaded tokenised cache {cache}")
    else:
        all_docs = build_docs(texts, golds, ls, piecer, shift, "fold1+2")
        test_docs = [dict(id=k, ids=a, cs=b, ce=c) for k, v in test_texts.items() for a, b, c in [piecer.encode_doc(v)]]
        tmp = f"{cache}.{os.getpid()}.tmp"
        with open(tmp, "wb") as f: pickle.dump((all_docs, test_docs), f)
        os.replace(tmp, cache)
    order = sorted(d["id"] for d in all_docs)
    random.Random(1234).shuffle(order)                                    # fixed split shared by ALL runs
    if args.n_folds > 0:                                                  # k-fold CV: dev = fold `fold`, train = the other folds
        assert 0 <= args.fold < args.n_folds
        dev_ids = set(np.array_split(np.array(order), args.n_folds)[args.fold].tolist())
        dev_docs = [d for d in all_docs if d["id"] in dev_ids]
    else:
        n_dev = max(1, int(len(order) * args.dev_frac))
        dev_ids = set(order[:n_dev])
        dev_docs = [d for d in all_docs if d["id"] in dev_ids][:args.dev_docs]
    train_docs = [d for d in all_docs if (d["id"] not in dev_ids or args.full_data)]
    if args.max_train_docs: train_docs = train_docs[:args.max_train_docs]
    print(f"  train docs={len(train_docs)} dev docs={len(dev_docs)} test docs={len(test_docs)}")
    print(f"  ORACLE strict macro-F1 of the tokenisation/alignment pipeline: train={oracle_f1(train_docs, ls):.4f} dev={oracle_f1(dev_docs, ls):.4f}")
    cnt = np.zeros(ls.n, np.int64)
    for d in train_docs: cnt += np.bincount(d["y"][d["y"] >= 0].astype(np.int64), minlength=ls.n)
    print("  token label counts:", {ls.names[i]: int(c) for i, c in enumerate(cnt) if i == 0 or i % 2 == 1})

    teacher = None
    if args.method == "kd":
        assert args.teacher, "--teacher runs/<full_run>/best.pt is required for kd"
        teacher, tmeta = load_ner_ckpt(args.teacher, device)
        layer_map = [int(round(x)) for x in np.linspace(0, tmeta["lm_cfg"]["n_layers"] - 1, args.student_layers)]
        print(f"  KD student = teacher blocks {layer_map}; teacher params={count_params(teacher)[0] / 1e6:.1f}M")
        lm, lm_cfg = load_phase1(args.init, layer_map=layer_map)
        args.attn = "causal" if tmeta["causal"] else "bidir"     # student sees the same context as the teacher
        args.head = tmeta.get("head", "linear")                  # so the teacher head can initialise the student head
    else:
        lm, lm_cfg = load_phase1(args.init)
    assert args.window <= lm_cfg["max_seq_len"], "window longer than the Phase-1 context (RoPE tables are built for 512)"
    model = NERModel(lm, ls.n, causal=(args.attn == "causal"), head=args.head)
    if args.method == "kd":
        with torch.no_grad():                                             # start student head from the teacher head
            model.head.load_state_dict(teacher.head.state_dict())
    before = {id(p) for p in model.parameters()}
    apply_method(model, args)
    mark_new_params_trainable(model, before)
    model.to(device)
    tot, trn = count_params(model)
    print(f"  params total={tot / 1e6:.2f}M trainable={trn / 1e6:.3f}M ({100 * trn / tot:.2f}%)")

    decay, no_decay, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (head_p if n.startswith("head.") else decay if p.ndim >= 2 else no_decay).append(p)
    groups = [dict(params=head_p, lr=args.lr_head, weight_decay=0.0), dict(params=no_decay, lr=base_lr, weight_decay=0.0),
              dict(params=decay, lr=base_lr, weight_decay=args.wd)]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    cw = torch.ones(ls.n, device=device); cw[0] = args.o_weight

    rng = random.Random(args.seed)
    meta = dict(method=args.method, causal=model.causal, labels=labels, lm_cfg=lm_cfg, head=args.head)
    best_path = os.path.join(args.out, "best.pt")
    wins = train_windows(train_docs, args.window, args.stride, args.neg_keep, rng)
    steps_per_epoch = max(1, len(wins) // args.bs)
    planned = int(args.epochs * steps_per_epoch)
    total, step, best_f1, best_step, t_bench = planned, 0, -1.0, 0, None
    eval_every = None
    print(f"  windows/epoch={len(wins)} steps/epoch={steps_per_epoch} planned steps={planned}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()

    def run_eval(tag):
        nonlocal best_f1, best_step
        pr, _ = predict_docs(model, dev_docs, ls, args.window, args.stride, device, ("viterbi",), amp=amp)
        sc = strict_scores(dev_truth_of(dev_docs), pr["viterbi"], labels)
        print(f"  [eval {tag}] step={step} dev strict macro-F1={sc['macro_F1']:.4f} P={sc['macro_P']:.4f} R={sc['macro_R']:.4f}", flush=True)
        if sc["macro_F1"] > best_f1:
            best_f1, best_step = sc["macro_F1"], step; save_ckpt(best_path, model, meta)
        model.train()

    model.train()
    t_start = time.time(); done = False; ep = 0; loss_ema = None
    while not done:
        order_w = train_windows(train_docs, args.window, args.stride, args.neg_keep, rng); rng.shuffle(order_w); ep += 1   # fresh negative-window sample each epoch
        for bi in range(0, len(order_w) - args.bs + 1, args.bs):
            if step == 5: t_bench = time.time()
            if step == 35 and t_bench is not None:                      # fix the schedule from the measured speed
                sps = (time.time() - t_bench) / 30
                left = args.time_budget_min * 60 * 0.80 - (time.time() - t_global)
                total = max(60, min(planned, 35 + int(left / sps)))
                eval_every = max(50, steps_per_epoch)                     # evaluate (and keep the best) once per epoch
                print(f"  speed={1 / sps:.2f} steps/s -> total steps={total} (planned {planned}); eval every {eval_every}", flush=True)
            if step >= total or (time.time() - t_global) > args.time_budget_min * 60 * 0.95: done = True; break
            batch = [(train_docs[di], a, b) for di, a, b in order_w[bi:bi + args.bs]]
            ids, lab, msk = collate([(d["ids"][a:b], d["y"][a:b]) for d, a, b in batch], device)
            lr_scale = lr_at(step, total, args.warmup, 1.0)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_scale
            with torch.autocast("cuda", enabled=amp):
                logits = model(ids, msk)
            lf = logits.float()
            ce = F.cross_entropy(lf.view(-1, ls.n), lab.view(-1), weight=cw, ignore_index=IGN)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", enabled=amp):
                    tl = teacher(ids, msk).float()
                m = lab.view(-1) != IGN
                T = args.kd_T
                kd = F.kl_div(F.log_softmax(lf.view(-1, ls.n)[m] / T, -1), F.softmax(tl.view(-1, ls.n)[m] / T, -1),
                              reduction="batchmean") * T * T
                loss = args.kd_alpha * ce + (1 - args.kd_alpha) * kd
            else:
                loss = ce
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            loss_ema = loss.item() if loss_ema is None else 0.98 * loss_ema + 0.02 * loss.item()
            if step % 50 == 0:
                print(f"  step {step}/{total} ep {ep} loss {loss_ema:.4f} lr {base_lr * lr_scale:.2e} "
                      f"{(time.time() - t_start) / 60:.1f}min", flush=True)
            if eval_every and step % eval_every == 0 and step < total: run_eval("mid")
    train_min = (time.time() - t_start) / 60
    run_eval("final")
    peak_gb = torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else 0.0

    ck = torch.load(best_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["state"]); model.to(device)
    model.eval()
    dev_lp, tps = collect_logp(model, dev_docs, args.window, args.stride, device, amp)
    np.savez_compressed(os.path.join(args.out, "dev_logp.npz"), **dev_lp)
    pr = {dm: decode_logp(dev_lp, dev_docs, ls, dm) for dm in ("greedy", "viterbi")}
    truth = dev_truth_of(dev_docs)
    sc = {dm: strict_scores(truth, pr[dm], labels) for dm in pr}
    use = max(sc, key=lambda k: sc[k]["macro_F1"])
    print(f"  dev macro-F1: greedy={sc['greedy']['macro_F1']:.4f} viterbi={sc['viterbi']['macro_F1']:.4f} -> use {use}")
    for l in labels:
        r = sc[use]["per_label"][l]; print(f"    {l:10s} P={r['P']:.3f} R={r['R']:.3f} F1={r['F1']:.3f}  (tp={r['tp']} fp={r['fp']} fn={r['fn']})")
    with open(os.path.join(args.out, "dev_truth.json"), "w") as f:
        json.dump({k: [[s, e - shift, l] for s, e, l in v] for k, v in truth.items()}, f)
    with open(os.path.join(args.out, "dev_pred.json"), "w") as f: json.dump(shifted(pr[use], shift), f)

    res = dict(method=args.method, attn=args.attn, params_total=tot, params_trainable=trn, train_minutes=train_min,
               steps=step, planned_steps=planned, peak_gpu_gb=peak_gb, infer_tokens_per_s=tps, best_step=best_step,
               dev_macro_F1=sc[use]["macro_F1"], dev_macro_P=sc[use]["macro_P"], dev_macro_R=sc[use]["macro_R"],
               dev_greedy_F1=sc["greedy"]["macro_F1"], dev_viterbi_F1=sc["viterbi"]["macro_F1"], decode=use,
               per_label=sc[use]["per_label"], errors=error_breakdown(truth, pr[use], labels), args=vars(args),
               shift=shift, fold=args.fold if args.n_folds > 0 else None, n_folds=args.n_folds, dev_ids=sorted(dev_ids), n_dev_docs=len(dev_docs))
    if not args.no_test:
        test_lp, _ = collect_logp(model, test_docs, args.window, args.stride, device, amp)
        np.savez_compressed(os.path.join(args.out, "test_logp.npz"), **test_lp)
        pred = shifted(decode_logp(test_lp, test_docs, ls, use), shift)
        assert set(pred) == {d["id"] for d in test_docs}
        pj = os.path.join(args.out, "prediction.json")
        with open(pj, "w") as f: json.dump(pred, f)
        with zipfile.ZipFile(os.path.join(args.out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
        res["n_test_docs"], res["n_test_entities"] = len(pred), sum(len(v) for v in pred.values())
        print(f"  wrote {args.out}/submission.zip  ({res['n_test_entities']} entities in {len(pred)} docs)")
    with open(os.path.join(args.out, "results.json"), "w") as f: json.dump(res, f, indent=1, default=str)
    print(f"== done in {(time.time() - t_global) / 60:.1f} min | dev strict macro-F1 = {res['dev_macro_F1']:.4f}")


if __name__ == "__main__":
    main()

Writing lner.py


In [2]:
%%writefile ensemble.py
#!/usr/bin/env python3
"""
ensemble.py -- combine k-fold runs / several methods, score them OUT-OF-FOLD on all labelled docs, post-process, write a submission.

  python ensemble.py --data_dir p2data --runs_dir runs --names lora,lorma --out ens_lora_lorma --grid
  python ensemble.py --data_dir p2data --runs_dir runs --names lora,lorma,full --out ens_all --min_conf 0.5 --consistency 1

A "name" is a run-family: runs/<name>_f0 ... runs/<name>_f4 (written by lner.py --n_folds 5 --fold k).
Out-of-fold (OOF) score of a family = every labelled doc is predicted by the fold-model that never saw it -> strict macro-F1 on all 70 docs.
Ensembling = average of token probabilities over all selected runs (OOF: over the families for that doc; test: over every run).
"""
import argparse, glob, json, os, pickle, re, zipfile
from collections import Counter, defaultdict
import numpy as np
import lner


def load_runs(runs_dir, name):
    dirs = sorted(glob.glob(os.path.join(runs_dir, f"{name}_f[0-9]*")))
    runs = []
    for d in dirs:
        if all(os.path.exists(os.path.join(d, f)) for f in ("results.json", "dev_logp.npz", "test_logp.npz")):
            runs.append(d)
    return runs


def avg_probs(list_of_lp):
    return np.log(np.maximum(np.mean([np.exp(x.astype(np.float32)) for x in list_of_lp], 0), 1e-9))


def decode_doc(lpr, csr, cer, ls, min_conf):
    path = lner.viterbi(lpr, ls)
    P = np.exp(lpr); out = []
    for i, j, k in lner.spans_from_ids(path):
        conf = float((P[i:j + 1, 1 + 2 * k] + P[i:j + 1, 2 + 2 * k]).mean())      # probability mass on label k (B or I)
        if conf >= min_conf: out.append((int(csr[i]), int(cer[j]), ls.labels[k], conf))
    return out


def consistency(text, spans, allowed, min_count=2, share=0.8, minlen=4):
    """If the same string was tagged with the same label (>=share of >=min_count times) add the untagged occurrences."""
    cnt = defaultdict(Counter)
    for s, e, l, _ in spans: cnt[text[s:e]][l] += 1
    covered = [(s, e) for s, e, _, _ in spans]; new = []
    for string, c in cnt.items():
        tot = sum(c.values()); l, top = c.most_common(1)[0]
        if len(string) < minlen or tot < min_count or top / tot < share or l not in allowed: continue
        for m in re.finditer(r"(?<!\w)" + re.escape(string) + r"(?!\w)", text):
            s, e = m.span()
            if any(s < ce and cs < e for cs, ce in covered): continue
            new.append((s, e, l, 1.0)); covered.append((s, e))
    return spans + new


def decode_all(lp_by_doc, docs, ls):
    """Viterbi-decode every doc once -> {id: [(start, end, label, conf)]} (conf = mean prob. mass on the span's label)."""
    out = {}
    for d in docs:
        lpr = lp_by_doc[d["id"]].astype(np.float32); real = d["cs"] >= 0
        out[d["id"]] = decode_doc(lpr, d["cs"][real], d["ce"][real], ls, 0.0) if len(lpr) else []
    return out


def edge_stats(texts, truth_all, labels):
    """For each label: how often each character is the first / last character of a GOLD span (from the labelled docs only)."""
    st = {l: dict(n=0, start=Counter(), end=Counter()) for l in labels}
    for did, spans in truth_all.items():
        t = texts[did]
        for s_, e_, l in spans:
            if e_ > s_ and l in st:
                st[l]["n"] += 1; st[l]["start"][t[s_]] += 1; st[l]["end"][t[e_ - 1]] += 1
    return st


def trim_span(text, s, e, label, stats, min_share=0.01):
    """Remove leading/trailing punctuation that gold spans of this label (almost) never start/end with."""
    st = stats.get(label)
    if not st or st["n"] < 20: return s, e
    ok_start = {c for c, n in st["start"].items() if n / st["n"] >= min_share}
    ok_end = {c for c, n in st["end"].items() if n / st["n"] >= min_share}
    while e > s and not text[e - 1].isalnum() and text[e - 1] not in ok_end: e -= 1
    while s < e and not text[s].isalnum() and text[s] not in ok_start: s += 1
    return s, e


def finalize(spans_by_doc, texts, thr, cons, cons_labels, trim=False, stats=None):
    """thr: float or {label: float}.  Order: confidence filter -> boundary trim -> same-string consistency."""
    out = {}
    for did, sp in spans_by_doc.items():
        keep = []
        for s_, e_, l, c in sp:
            t = thr.get(l, 0.0) if isinstance(thr, dict) else thr
            if c < t: continue
            if trim:
                s_, e_ = trim_span(texts[did], s_, e_, l, stats)
                if e_ <= s_: continue
            keep.append((s_, e_, l, c))
        if cons: keep = consistency(texts[did], keep, cons_labels)
        out[did] = [(s_, e_, l) for s_, e_, l, _ in keep]
    return out


THR_GRID = (0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)


def tune_thresholds(spans, ids, texts, truth, labels, cons, cons_labels, trim, stats, passes=2, min_gain=1e-3):
    """Coordinate ascent on per-label confidence thresholds (a move must gain > min_gain macro-F1, to limit over-fitting)."""
    sub = {i: spans[i] for i in ids}; tr = {i: truth[i] for i in ids}
    score = lambda t: lner.strict_scores(tr, finalize(sub, texts, t, cons, cons_labels, trim, stats), labels)["macro_F1"]
    thr = {l: 0.0 for l in labels}; best = score(thr)
    for _ in range(passes):
        for l in labels:
            for g in THR_GRID:
                if g == thr[l]: continue
                t = dict(thr); t[l] = g; sc = score(t)
                if sc > best + min_gain: best, thr = sc, t
    return thr, best


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True); ap.add_argument("--runs_dir", required=True)
    ap.add_argument("--names", required=True, help="comma-separated run families, e.g. lora,lorma,full")
    ap.add_argument("--out", required=True)
    ap.add_argument("--min_conf", type=float, default=0.0); ap.add_argument("--consistency", type=int, default=0)
    ap.add_argument("--cons_labels", default="APP,RESP,COURT,AUTH,JUDGE,WIT,A.COUNSEL,R.COUNSEL,STAT,PREC")
    ap.add_argument("--final_names", default="", help="comma-separated families trained on ALL labelled docs: used for the TEST average only (their dev_logp is not out-of-fold)")
    ap.add_argument("--final_weight", type=float, default=1.0, help="weight of each final run relative to a CV run in the test average")
    ap.add_argument("--trim", type=int, default=0, help="1 = strip punctuation at span edges that gold spans of that label never have")
    ap.add_argument("--tune", type=int, default=0, help="1 = tune a confidence threshold PER LABEL on the OOF predictions (also prints an honest split-half check)")
    ap.add_argument("--cache", default=None, help="tokcache_*.pkl to use (default: newest in data_dir)")
    ap.add_argument("--grid", action="store_true", help="print OOF F1 for several post-processing settings, then use the flags above")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    labels, texts, golds, test_texts = lner.load_split_files(args.data_dir)
    ls = lner.LabelSpace(labels)
    cache = args.cache or sorted(glob.glob(os.path.join(args.data_dir, "tokcache_*.pkl")), key=os.path.getmtime)[-1]
    print("  token cache:", cache)
    with open(cache, "rb") as f: all_docs, test_docs = pickle.load(f)
    by_id = {d["id"]: d for d in all_docs}
    names = [n.strip() for n in args.names.split(",") if n.strip()]
    finals = [n.strip() for n in args.final_names.split(",") if n.strip()]
    cv_names = [n for n in names if n not in finals]
    assert cv_names, "need at least one cross-validated (out-of-fold) family in --names"
    all_names = cv_names + [n for n in finals if n not in cv_names]
    fam = {n: load_runs(args.runs_dir, n) for n in all_names}
    for n, r in fam.items(): print(f"  family {n}: {len(r)} runs" + ("   [FINAL: test average only]" if n in finals else ""))
    assert all(fam.values()), "a family has no finished runs"
    shift = json.load(open(os.path.join(fam[cv_names[0]][0], "results.json")))["shift"]
    truth_all = {d["id"]: sorted(set(d["ents"])) for d in all_docs}
    stats = edge_stats(texts, truth_all, labels)

    oof = {}
    for n in cv_names:
        oof[n] = {}
        for r in fam[n]:
            z = np.load(os.path.join(r, "dev_logp.npz"))
            for k in z.files:
                oof[n][k] = z[k]
                assert len(z[k]) == int((by_id[k]["cs"] >= 0).sum()), (f"{r}: token count differs from the cache ({len(z[k])} vs {int((by_id[k]['cs'] >= 0).sum())}) -- "
                                                          "families trained with DIFFERENT tokenizers cannot be averaged; pass --cache of the matching tokenizer or ensemble them separately")
        print(f"  {n}: OOF coverage {len(oof[n])}/{len(all_docs)} docs")
    common = sorted(set.intersection(*[set(v) for v in oof.values()]))
    docs_c = [by_id[i] for i in common]
    truth = {i: truth_all[i] for i in common}

    print(f"\nOUT-OF-FOLD strict macro-F1 on {len(common)} labelled docs (viterbi, no post-processing)")
    rows = []
    for n in cv_names:
        pr = lner.decode_logp({i: oof[n][i] for i in common}, docs_c, ls, "viterbi")
        sc = lner.strict_scores(truth, pr, labels); rows.append((n, sc))
        print(f"  {n:14s} F1={sc['macro_F1']:.4f}  P={sc['macro_P']:.4f}  R={sc['macro_R']:.4f}")
    ens_oof = {i: avg_probs([oof[n][i] for n in cv_names]) for i in common}
    spans_oof = decode_all(ens_oof, docs_c, ls)
    sc = lner.strict_scores(truth, finalize(spans_oof, texts, 0.0, 0, set()), labels)
    print(f"  {'ENSEMBLE':14s} F1={sc['macro_F1']:.4f}  P={sc['macro_P']:.4f}  R={sc['macro_R']:.4f}")
    cons_labels = set(x.strip() for x in args.cons_labels.split(","))
    S = lambda thr, cn, tr, ids=None: lner.strict_scores({i: truth[i] for i in (ids or common)},
            finalize({i: spans_oof[i] for i in (ids or common)}, texts, thr, cn, cons_labels, bool(tr), stats), labels)

    if args.grid:
        print("\nPost-processing grid on the OOF ensemble  -> macro-F1 / P / R")
        for tr in (0, 1):
            for mc in (0.0, 0.3, 0.4, 0.5, 0.6, 0.7):
                for cn in (0, 1):
                    s2 = S(mc, cn, tr)
                    print(f"  trim={tr} min_conf={mc:.1f} consistency={cn}  F1={s2['macro_F1']:.4f} P={s2['macro_P']:.4f} R={s2['macro_R']:.4f}")
    thr = args.min_conf
    if args.tune:
        thr, tuned = tune_thresholds(spans_oof, common, texts, truth, labels, args.consistency, cons_labels, args.trim, stats)
        print("\nper-label thresholds tuned on ALL OOF docs:", {l: t for l, t in thr.items() if t > 0}, f"-> OOF F1 {tuned:.4f} (optimistic: tuned on the same docs)")
        half = [common[0::2], common[1::2]]; gains = []
        for a, b in ((0, 1), (1, 0)):                                   # honest check: tune on one half of the docs, score on the other
            th, _ = tune_thresholds(spans_oof, half[a], texts, truth, labels, args.consistency, cons_labels, args.trim, stats)
            base = S(args.min_conf, args.consistency, args.trim, half[b])["macro_F1"]; got = S(th, args.consistency, args.trim, half[b])["macro_F1"]
            gains.append(got - base); print(f"  split-half check {a}->{b}: held-out F1 {base:.4f} -> {got:.4f}  ({got - base:+.4f})")
        print(f"  mean held-out gain of per-label tuning: {np.mean(gains):+.4f}   (if <= ~0 or negative, DO NOT use --tune)")
    final_oof = S(thr, args.consistency, args.trim)
    print(f"\nchosen: min_conf={args.min_conf} trim={args.trim} consistency={args.consistency} tune={args.tune} -> OOF macro-F1 {final_oof['macro_F1']:.4f}")
    for l in labels:
        r = final_oof["per_label"][l]; print(f"    {l:10s} P={r['P']:.3f} R={r['R']:.3f} F1={r['F1']:.3f}")

    acc, wsum = {}, 0.0
    for n in all_names:
        w = args.final_weight if n in finals else 1.0
        for r in fam[n]:
            z = np.load(os.path.join(r, "test_logp.npz")); wsum += w
            for k in z.files:
                acc[k] = acc.get(k, 0) + w * np.exp(z[k].astype(np.float32))
    cnt = sum(len(fam[n]) for n in all_names)
    test_lp = {k: np.log(np.maximum(v / wsum, 1e-9)) for k, v in acc.items()}
    spans_test = decode_all(test_lp, test_docs, ls)
    pred = finalize(spans_test, test_texts, thr, args.consistency, cons_labels, bool(args.trim), stats)
    pred = {k: [[s, e - shift, l] for s, e, l in v] for k, v in pred.items()}
    assert set(pred) == {d["id"] for d in test_docs}
    pj = os.path.join(args.out, "prediction.json")
    json.dump(pred, open(pj, "w"))
    with zipfile.ZipFile(os.path.join(args.out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    json.dump(dict(names=names, final_names=finals, n_runs=cnt, oof_ensemble_F1=final_oof["macro_F1"], per_family_oof={n: s["macro_F1"] for n, s in rows},
                   min_conf=args.min_conf, thresholds=thr if isinstance(thr, dict) else None, trim=args.trim, consistency=args.consistency,
                   n_test_entities=sum(len(v) for v in pred.values()), per_label=final_oof["per_label"]), open(os.path.join(args.out, "ensemble_report.json"), "w"), indent=1)
    print(f"wrote {args.out}/submission.zip from {cnt} runs ({sum(len(v) for v in pred.values())} entities in {len(pred)} docs)")


if __name__ == "__main__":
    main()

Writing ensemble.py


In [3]:
import os, sys, json, time, glob, subprocess, shutil, collections
import numpy as np, pandas as pd, torch
WORK = "/kaggle/working"
def find(name, root="/kaggle/input"):
    return sorted([os.path.join(d, name) for d, _, fs in os.walk(root, followlinks=True) if name in fs], key=os.path.getmtime)
INIT = (find("model_step45000.pt") or find("model.pt") or [None])[-1]
TOK  = (find("bpe_tokenizer.json") or [None])[-1]
assert INIT and TOK, f"attach your Phase-1 decoder output dataset. /kaggle/input has: {os.listdir('/kaggle/input')}"
print("INIT:", INIT, "\nTOK :", TOK)
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, "NO GPU! Settings -> Accelerator -> GPU T4 x2, then restart. On CPU this notebook is ~100x too slow."
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

INIT: /kaggle/input/notebooks/parth1223/decoder-part-3/cs781_phase1/rope_mqa_swiglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model_step45000.pt 
TOK : /kaggle/input/notebooks/parth1223/decoder-part-3/cs781_phase1/rope_mqa_swiglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/bpe_tokenizer.json
GPUs: 2 ['Tesla T4', 'Tesla T4']


## 1. Data

In [4]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HuggingFace")
DATA = f"{WORK}/p2data"; os.makedirs(DATA, exist_ok=True)
FILES = {"Phase_2_LNER/Train/fold1-text.json":"fold1-text.json","Phase_2_LNER/Train/fold1-gold.json":"fold1-gold.json",
         "Phase_2_LNER/Train/fold2-text.json":"fold2-text.json","Phase_2_LNER/Train/fold2-gold.json":"fold2-gold.json",
         "Phase_2_LNER/Train/labels.txt":"labels.txt","Phase_2_LNER/Test/fold3-text.json":"fold3-text.json"}
for remote, local in FILES.items():
    if not os.path.exists(f"{DATA}/{local}"):
        shutil.copy(hf_hub_download("Exploration-Lab/CS-781-Capstone", remote, repo_type="dataset", token=hf_token), f"{DATA}/{local}")
print(sorted(os.listdir(DATA)))

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

['fold1-gold.json', 'fold1-text.json', 'fold2-gold.json', 'fold2-text.json', 'fold3-text.json', 'labels.txt']


## 2. Helpers (CV launcher, final-model launcher, waiting, tables)

In [5]:
N_FOLDS = 5
def launch_cv(name, gpu, extra, epochs, folds=None, budget=25):
    folds = list(range(N_FOLDS)) if folds is None else folds
    cmds = []
    for k in folds:
        out = f"{WORK}/runs/{name}_f{k}"; os.makedirs(out, exist_ok=True)
        cmds.append(f"PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} python lner.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {out} "
                    f"--n_folds {N_FOLDS} --fold {k} --epochs {epochs} --time_budget_min {budget} {extra} > {out}/log.txt 2>&1")
    return subprocess.Popen(" ; ".join(cmds), shell=True), name

def launch_final(name, gpu, extra, epochs, seeds, budget=30):
    """Train on ALL labelled docs (no held-out fold). Run k of the family uses seed seeds[k]; there is no dev score to trust for these."""
    cmds = []
    for k, sd in enumerate(seeds):
        out = f"{WORK}/runs/{name}_f{k}"; os.makedirs(out, exist_ok=True)
        cmds.append(f"PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} python lner.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {out} "
                    f"--full_data --seed {sd} --epochs {epochs} --time_budget_min {budget} {extra} > {out}/log.txt 2>&1")
    return subprocess.Popen(" ; ".join(cmds), shell=True), name

def wait(jobs, every=120):
    while any(p.poll() is None for p, _ in jobs):
        time.sleep(every)
        for p, n in jobs:
            logs = sorted(glob.glob(f"{WORK}/runs/{n}_f*/log.txt"), key=os.path.getmtime)
            tail = open(logs[-1]).read().strip().splitlines()[-1:] if logs else []
            print(f"[{n}] {'running' if p.poll() is None else 'done'} | {logs[-1].split('/')[-2] if logs else ''} | {tail[0][:120] if tail else ''}")
    for n in sorted({n for _, n in jobs}):
        print(f"== {n}: {len(glob.glob(f'{WORK}/runs/{n}_f*/test_logp.npz'))} runs finished")

def fam_table(names):
    rows = []
    for n in names:
        for d in sorted(glob.glob(f"{WORK}/runs/{n}_f*/results.json")):
            x = json.load(open(d)); rows.append(dict(run=d.split("/")[-2], dev_F1=round(x["dev_macro_F1"], 4), best_step=x["best_step"], steps=x["steps"],
                                                     best_frac=round(x["best_step"] / max(1, x["steps"]), 2), train_min=round(x["train_minutes"], 1), n_dev_docs=x["n_dev_docs"]))
    return pd.DataFrame(rows)

def run_ens(out, names, extra="", show=4500):
    r = subprocess.run(f"python ensemble.py --data_dir {DATA} --runs_dir {WORK}/runs --names {names} --out {WORK}/ens/{out} {extra}", shell=True, capture_output=True, text=True)
    print(r.stdout[-show:], r.stderr[-1500:])
    z = f"{WORK}/ens/{out}/submission.zip"
    if os.path.exists(z): os.makedirs(f"{WORK}/submissions_v3", exist_ok=True); shutil.copy(z, f"{WORK}/submissions_v3/{out}.zip")

## 3. Build the tokenisation cache (1 tiny run) and import your v2 CV runs (only the small `results/dev_logp/test_logp` files, not the checkpoints)

In [6]:
os.makedirs(f"{WORK}/runs/smoke_f0", exist_ok=True)
subprocess.run(f"python lner.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {WORK}/runs/smoke_f0 --method lora --max_train_docs 8 --epochs 1 --time_budget_min 4 --no_test > {WORK}/runs/smoke_f0/log.txt 2>&1", shell=True)
print("".join(l for l in open(f"{WORK}/runs/smoke_f0/log.txt") if any(k in l for k in ["train docs", "ORACLE", "Traceback", "Error"])))
assert glob.glob(f"{DATA}/tokcache_*.pkl"), "tokenisation cache missing -> read runs/smoke_f0/log.txt"

V2 = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/runs/*_f0/results.json", recursive=True)})
print("v2 run folders found:", V2)
IMPORTED = []
for root in V2:
    for d in sorted(glob.glob(f"{root}/*_f[0-9]*")):
        if not os.path.exists(f"{d}/test_logp.npz"): continue
        fam = os.path.basename(d).rsplit("_f", 1)[0]
        if "lora" not in fam and "lorma" not in fam: continue            # LoRA / LoRMA families only
        dst = f"{WORK}/runs/{os.path.basename(d)}"; os.makedirs(dst, exist_ok=True)
        for f in ("results.json", "dev_logp.npz", "test_logp.npz"): shutil.copy(f"{d}/{f}", dst)
        IMPORTED.append(fam)
IMPORTED = sorted(set(IMPORTED)); print("imported families:", IMPORTED)
assert IMPORTED, "no v2 runs found: attach your v2 notebook output as an input (it must contain runs/lora_f0 ...)"

  train docs=8 dev docs=4 test docs=35
  ORACLE strict macro-F1 of the tokenisation/alignment pipeline: train=0.9953 dev=0.9991

v2 run folders found: ['/kaggle/input/notebooks/parth1223/decoder-phase-2-finetuning-part-2-2/runs']
imported families: ['lora', 'lora_causal', 'lorma', 'lorma_causal']


## 4. STEP 1 — re-ensemble the v2 runs with the new post-processing (a few minutes, no training)
Read the grid: `trim=1` rows vs `trim=0` rows tell you whether boundary errors are fixable; `--tune 1` prints a split-half check — **if its mean held-out gain is ~0 or negative, ignore per-label thresholds**.

In [7]:
FAM = ",".join(IMPORTED)
run_ens("v3_step1_grid", FAM, "--grid --trim 1 --tune 1")

  token cache: /kaggle/working/p2data/tokcache_258331cd7f.pkl
  family lora: 5 runs
  family lora_causal: 5 runs
  family lorma: 5 runs
  family lorma_causal: 5 runs
  lora: OOF coverage 70/70 docs
  lora_causal: OOF coverage 70/70 docs
  lorma: OOF coverage 70/70 docs
  lorma_causal: OOF coverage 70/70 docs

OUT-OF-FOLD strict macro-F1 on 70 labelled docs (viterbi, no post-processing)
  lora           F1=0.4947  P=0.4933  R=0.5063
  lora_causal    F1=0.4480  P=0.4451  R=0.4568
  lorma          F1=0.4980  P=0.4998  R=0.5050
  lorma_causal   F1=0.4544  P=0.4470  R=0.4669
  ENSEMBLE       F1=0.5242  P=0.5421  R=0.5195

Post-processing grid on the OOF ensemble  -> macro-F1 / P / R
  trim=0 min_conf=0.0 consistency=0  F1=0.5242 P=0.5421 R=0.5195
  trim=0 min_conf=0.0 consistency=1  F1=0.5317 P=0.5319 R=0.5404
  trim=0 min_conf=0.3 consistency=0  F1=0.5238 P=0.5421 R=0.5190
  trim=0 min_conf=0.3 consistency=1  F1=0.5313 P=0.5316 R=0.5399
  trim=0 min_conf=0.4 consistency=0  F1=0.5246 P=0.54

Pick settings **from the OOF numbers above** (not from Codabench), then write the zip:

In [8]:
MIN_CONF, TRIM, CONSIST, TUNE = 0.0, 1, 0, 0          # <- set from the grid / split-half check above
run_ens("v3_cv_v2", FAM, f"--min_conf {MIN_CONF} --trim {TRIM} --consistency {CONSIST} --tune {TUNE}")

  token cache: /kaggle/working/p2data/tokcache_258331cd7f.pkl
  family lora: 5 runs
  family lora_causal: 5 runs
  family lorma: 5 runs
  family lorma_causal: 5 runs
  lora: OOF coverage 70/70 docs
  lora_causal: OOF coverage 70/70 docs
  lorma: OOF coverage 70/70 docs
  lorma_causal: OOF coverage 70/70 docs

OUT-OF-FOLD strict macro-F1 on 70 labelled docs (viterbi, no post-processing)
  lora           F1=0.4947  P=0.4933  R=0.5063
  lora_causal    F1=0.4480  P=0.4451  R=0.4568
  lorma          F1=0.4980  P=0.4998  R=0.5050
  lorma_causal   F1=0.4544  P=0.4470  R=0.4669
  ENSEMBLE       F1=0.5242  P=0.5421  R=0.5195

chosen: min_conf=0.0 trim=1 consistency=0 tune=0 -> OOF macro-F1 0.5274
    APP        P=0.213 R=0.125 F1=0.158
    RESP       P=0.306 R=0.155 F1=0.206
    A.COUNSEL  P=0.536 R=0.605 F1=0.569
    R.COUNSEL  P=0.579 R=0.614 F1=0.596
    JUDGE      P=0.587 R=0.649 F1=0.616
    WIT        P=0.296 R=0.171 F1=0.217
    AUTH       P=0.446 R=0.267 F1=0.334
    COURT      P=0.778 

## 5. STEP 2 (optional, ~70 min) — second CV round: different seed + hyper-parameters (YOUR knobs, all untuned)
Window 512 / stride 256 (more overlap), rank 32, lower LR, a heavier weight on the 'O' class (0.7, fewer spurious spans), seed 7, LoRA dropout 0.1.
If these families score about the same OOF as v2 -> you are on a plateau and the remaining lever is more data (step 3).

In [9]:
RUN_B = True
EP_B = 10
if RUN_B:
    jobs = [launch_cv("lora_b",  0, "--method lora  --attn bidir --rank 32 --alpha 64 --lora_dropout 0.1 --lr 5e-4 --o_weight 0.7 --seed 7 --window 512 --stride 256", EP_B),
            launch_cv("lorma_b", 1, "--method lorma --attn bidir --rank 32 --alpha 32 --lr 5e-4 --o_weight 0.7 --seed 7 --window 512 --stride 256", EP_B)]
    wait(jobs); print(fam_table(["lora_b", "lorma_b"]).to_string(index=False))

[lora_b] running | lora_b_f0 |   [eval mid] step=314 dev strict macro-F1=0.3558 P=0.3340 R=0.4006
[lorma_b] running | lorma_b_f0 |   step 350/3140 ep 2 loss 0.1364 lr 4.92e-04 1.8min
[lora_b] running | lora_b_f0 |   step 650/3140 ep 3 loss 0.0776 lr 4.61e-04 3.9min
[lorma_b] running | lorma_b_f0 |   step 750/3140 ep 3 loss 0.0794 lr 4.46e-04 3.9min
[lora_b] running | lora_b_f0 |   step 950/3140 ep 4 loss 0.0555 lr 4.10e-04 5.7min
[lorma_b] running | lorma_b_f0 |   step 1100/3140 ep 4 loss 0.0470 lr 3.78e-04 5.8min
[lora_b] running | lora_b_f0 |   step 1300/3140 ep 5 loss 0.0275 lr 3.31e-04 7.9min
[lorma_b] running | lorma_b_f0 |   step 1500/3140 ep 5 loss 0.0262 lr 2.81e-04 7.9min
[lora_b] running | lora_b_f0 |   step 1600/3140 ep 6 loss 0.0173 lr 2.55e-04 9.7min
[lorma_b] running | lorma_b_f0 |   step 1850/3140 ep 6 loss 0.0178 lr 1.91e-04 9.8min
[lora_b] running | lora_b_f0 |   step 1950/3140 ep 7 loss 0.0060 lr 1.67e-04 11.8min
[lorma_b] running | lorma_b_f0 |   step 2200/3140 ep 8 

In [10]:
CV_FAMS = [n for n in IMPORTED + ["lora_b", "lorma_b"] if len(glob.glob(f"{WORK}/runs/{n}_f*/test_logp.npz")) == N_FOLDS]
print("CV families:", CV_FAMS)
run_ens("v3_step2_grid", ",".join(CV_FAMS), "--grid --trim 1 --tune 1")

CV families: ['lora', 'lora_causal', 'lorma', 'lorma_causal', 'lora_b', 'lorma_b']
  token cache: /kaggle/working/p2data/tokcache_258331cd7f.pkl
  family lora: 5 runs
  family lora_causal: 5 runs
  family lorma: 5 runs
  family lorma_causal: 5 runs
  family lora_b: 5 runs
  family lorma_b: 5 runs
  lora: OOF coverage 70/70 docs
  lora_causal: OOF coverage 70/70 docs
  lorma: OOF coverage 70/70 docs
  lorma_causal: OOF coverage 70/70 docs
  lora_b: OOF coverage 70/70 docs
  lorma_b: OOF coverage 70/70 docs

OUT-OF-FOLD strict macro-F1 on 70 labelled docs (viterbi, no post-processing)
  lora           F1=0.4947  P=0.4933  R=0.5063
  lora_causal    F1=0.4480  P=0.4451  R=0.4568
  lorma          F1=0.4980  P=0.4998  R=0.5050
  lorma_causal   F1=0.4544  P=0.4470  R=0.4669
  lora_b         F1=0.5004  P=0.5091  R=0.5007
  lorma_b        F1=0.4990  P=0.4983  R=0.5072
  ENSEMBLE       F1=0.5358  P=0.5589  R=0.5274

Post-processing grid on the OOF ensemble  -> macro-F1 / P / R
  trim=0 min_conf=

## 6. STEP 3 (~50 min) — final models on ALL 70 labelled docs (3 seeds x LoRA on GPU0, 3 seeds x LoRMA₊ on GPU1)
`best_frac` in the tables above (best checkpoint position / total steps) tells you how long the CV runs needed. The final runs have no held-out docs, so **FIN_EPOCHS is your decision**:
too many epochs memorise the 70 docs (train loss reaches ~0.004 in your hybrid run), too few under-fit. Default 8 = a bit below the v2 setting.

In [11]:
v2t = fam_table([n for n in IMPORTED if "causal" not in n and not n.endswith("_b")])
print(v2t.groupby(v2t.run.str.rsplit("_f", n=1).str[0])[["best_frac", "dev_F1"]].median().round(2))
FIN_EPOCHS = 8                         # <- your decision
FIN_SEEDS = [101, 102, 103]
RUN_FINALS = True
if RUN_FINALS:
    jobs = [launch_final("fin_lora",  0, f"--method lora  --attn bidir --rank {16} --alpha {32}", FIN_EPOCHS, FIN_SEEDS),
            launch_final("fin_lorma", 1, f"--method lorma --attn bidir --rank {16} --alpha {16}", FIN_EPOCHS, FIN_SEEDS)]
    wait(jobs)

       best_frac  dev_F1
run                     
lora         0.8    0.50
lorma        1.0    0.51
[fin_lora] running | fin_lora_f0 |   step 350/2168 ep 2 loss 0.1782 lr 9.65e-04 1.9min
[fin_lorma] running | fin_lorma_f0 |   step 400/2168 ep 2 loss 0.1692 lr 9.49e-04 1.8min
[fin_lora] running | fin_lora_f0 |   step 700/2168 ep 3 loss 0.1017 lr 8.07e-04 3.8min
[fin_lorma] running | fin_lorma_f0 |   [eval mid] step=813 dev strict macro-F1=0.7268 P=0.7333 R=0.7562
[fin_lora] running | fin_lora_f0 |   step 1050/2168 ep 4 loss 0.0675 lr 5.64e-04 5.8min
[fin_lorma] running | fin_lorma_f0 |   step 1200/2168 ep 5 loss 0.0583 lr 4.51e-04 5.7min
[fin_lora] running | fin_lora_f0 |   step 1400/2168 ep 6 loss 0.0269 lr 3.04e-04 7.8min
[fin_lorma] running | fin_lorma_f0 |   [eval mid] step=1626 dev strict macro-F1=0.9541 P=0.9364 R=0.9733
[fin_lora] running | fin_lora_f0 |   step 1750/2168 ep 7 loss 0.0102 lr 9.79e-05 9.7min
[fin_lorma] running | fin_lorma_f0 |   step 2000/2168 ep 8 loss 0.0107 lr 

## 7. STEP 4 — the zips to submit
* `v3_cv_all` : only models that never saw the held-out fold (OOF-validated settings)
* `v3_cv_plus_fin` : CV models + final models (final runs get weight `FIN_W` each in the test average)
* `v3_fin_heavy` : final models dominate (weight 100)
Final models cannot be scored locally. Judge the three by (1) the OOF numbers of their CV part, (2) Codabench — but do not pick purely on a 0.003 leaderboard difference; with 35 test docs that is noise.

In [12]:
FIN_W = 2.0
FINS = [n for n in ["fin_lora", "fin_lorma"] if glob.glob(f"{WORK}/runs/{n}_f*/test_logp.npz")]
flags = f"--min_conf {MIN_CONF} --trim {TRIM} --consistency {CONSIST} --tune {TUNE}"
run_ens("v3_cv_all", ",".join(CV_FAMS), flags, show=1800)
if FINS:
    run_ens("v3_cv_plus_fin", ",".join(CV_FAMS + FINS), f"{flags} --final_names {','.join(FINS)} --final_weight {FIN_W}", show=1800)
    run_ens("v3_fin_heavy",   ",".join(CV_FAMS + FINS), f"{flags} --final_names {','.join(FINS)} --final_weight 100", show=1800)
print(sorted(os.listdir(f"{WORK}/submissions_v3")))

  token cache: /kaggle/working/p2data/tokcache_258331cd7f.pkl
  family lora: 5 runs
  family lora_causal: 5 runs
  family lorma: 5 runs
  family lorma_causal: 5 runs
  family lora_b: 5 runs
  family lorma_b: 5 runs
  lora: OOF coverage 70/70 docs
  lora_causal: OOF coverage 70/70 docs
  lorma: OOF coverage 70/70 docs
  lorma_causal: OOF coverage 70/70 docs
  lora_b: OOF coverage 70/70 docs
  lorma_b: OOF coverage 70/70 docs

OUT-OF-FOLD strict macro-F1 on 70 labelled docs (viterbi, no post-processing)
  lora           F1=0.4947  P=0.4933  R=0.5063
  lora_causal    F1=0.4480  P=0.4451  R=0.4568
  lorma          F1=0.4980  P=0.4998  R=0.5050
  lorma_causal   F1=0.4544  P=0.4470  R=0.4669
  lora_b         F1=0.5004  P=0.5091  R=0.5007
  lorma_b        F1=0.4990  P=0.4983  R=0.5072
  ENSEMBLE       F1=0.5358  P=0.5589  R=0.5274

chosen: min_conf=0.0 trim=1 consistency=0 tune=0 -> OOF macro-F1 0.5370
    APP        P=0.227 R=0.140 F1=0.173
    RESP       P=0.317 R=0.155 F1=0.208
    A.COUNS

## 8. Diagnostics for the report: why are APP / RESP / WIT weak?
Look at where those spans sit and what they get confused with. If the APP/RESP gold spans are mostly in the document header while the model also tags the same names later in the body (or the reverse), that is a **labelling-convention** effect no model change fixes — and a strong paragraph for your analysis.

In [13]:
from collections import Counter
labels = open(f"{DATA}/labels.txt").read().split()
T, G = {}, {}
for f in (1, 2):
    T.update(json.load(open(f"{DATA}/fold{f}-text.json"))); G.update(json.load(open(f"{DATA}/fold{f}-gold.json")))
rows = []
for l in labels:
    sp = [(d, s, e, t) for d, ents in G.items() for s, e, ll, *t in ents if ll == l]
    rel = [s / len(T[d]) for d, s, e, t in sp]
    first = sum(1 for d, s, e, t in sp if s < 1500)
    docs = len({d for d, *_ in sp}); strs = Counter(T[d][s:e] for d, s, e, t in sp)
    rows.append(dict(label=l, n_spans=len(sp), docs_with=docs, median_rel_pos=round(float(np.median(rel)), 3) if rel else None,
                     pct_in_first_1500_chars=round(100 * first / max(1, len(sp)), 1), distinct_strings=len(strs), top_string=strs.most_common(1)[0] if strs else None))
print(pd.DataFrame(rows).to_string(index=False))
rep = json.load(open(f"{WORK}/ens/v3_cv_all/ensemble_report.json"))
print(pd.DataFrame(rep["per_label"]).T[["P", "R", "F1"]].round(3).to_string())

    label  n_spans  docs_with  median_rel_pos  pct_in_first_1500_chars  distinct_strings              top_string
      APP      407         63           0.153                     28.5               142            (Merlin, 25)
     RESP      381         63           0.194                     26.5               110         (Custodian, 60)
A.COUNSEL      195         49           0.168                     31.3               125             (Ghosh, 10)
R.COUNSEL      184         47           0.243                     25.5               126     (Bhimasankaram, 19)
    JUDGE      245         66           0.271                     41.2               156    (S. ABDUL NAZEER, 6)
      WIT      187         12           0.307                      7.0                58     (Prakashi Devi, 27)
     AUTH     1003         43           0.525                      5.2               125 (State Government, 119)
    COURT     1013         66           0.426                      6.9                78       (

**Rules reminder.** Hyper-parameters, the choice of post-processing and the analysis are yours; declare the LLM assistance you used. Upload every zip in `submissions_v3/` to Codabench.